# EDA - User Trust Score Project

Notebook này dùng để khám phá dữ liệu IEEE-CIS Fraud Detection và tạo một số hình phục vụ báo cáo.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
FIGURE_DIR = '../reports/figures'
os.makedirs(FIGURE_DIR, exist_ok=True)


## 1. Đọc dữ liệu

Đặt `train_transaction.csv` và `train_identity.csv` vào thư mục `data/raw/` trước khi chạy notebook.

In [ ]:
train_transaction_path = '../data/raw/train_transaction.csv'
train_identity_path = '../data/raw/train_identity.csv'

train_transaction = pd.read_csv(train_transaction_path)
train_identity = pd.read_csv(train_identity_path)

print('train_transaction:', train_transaction.shape)
print('train_identity:', train_identity.shape)
train_transaction.head()


## 2. Phân bố nhãn gian lận

In [ ]:
target_counts = train_transaction['isFraud'].value_counts().sort_index()
target_ratio = train_transaction['isFraud'].value_counts(normalize=True).sort_index()

summary = pd.DataFrame({
    'count': target_counts,
    'ratio': target_ratio
})
summary.index = ['Non-fraud', 'Fraud']
summary


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(summary.index, summary['count'])
ax.set_title('Phân bố nhãn isFraud')
ax.set_ylabel('Số lượng')
fig.tight_layout()
fig.savefig(os.path.join(FIGURE_DIR, 'target_distribution.png'), dpi=160)
plt.show()


## 3. Phân phối số tiền giao dịch

In [ ]:
amount_clip = train_transaction['TransactionAmt'].clip(upper=train_transaction['TransactionAmt'].quantile(0.99))
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(amount_clip, bins=40)
ax.set_title('Phân phối TransactionAmt')
ax.set_xlabel('TransactionAmt')
ax.set_ylabel('Số lượng')
fig.tight_layout()
fig.savefig(os.path.join(FIGURE_DIR, 'transaction_amount_distribution.png'), dpi=160)
plt.show()


## 4. Tỷ lệ thiếu dữ liệu

In [ ]:
missing_ratio = train_transaction.isnull().mean().sort_values(ascending=False).head(20)
missing_ratio.to_frame('missing_ratio')


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(missing_ratio.index[::-1], missing_ratio.values[::-1])
ax.set_title('Top 20 cột thiếu dữ liệu nhiều nhất')
ax.set_xlabel('Missing ratio')
fig.tight_layout()
fig.savefig(os.path.join(FIGURE_DIR, 'missing_values_top20.png'), dpi=160)
plt.show()


## 5. Merge dữ liệu và thống kê nhanh

In [ ]:
df = train_transaction.merge(train_identity, on='TransactionID', how='left')
print('Merged shape:', df.shape)
print('Number of columns:', df.shape[1])
print('Number of rows:', df.shape[0])


## 6. Gợi ý nhận xét đưa vào báo cáo

- Dữ liệu có mất cân bằng lớp vì số giao dịch gian lận thường ít hơn nhiều so với giao dịch bình thường.
- Dữ liệu có nhiều cột bị thiếu, vì vậy cần loại bỏ cột thiếu quá nhiều và điền giá trị thiếu cho các cột còn lại.
- `TransactionAmt` có phân phối lệch phải, một số giao dịch có giá trị rất lớn.
- Do nhiều feature đã được ẩn danh, mô hình phù hợp để dự đoán nhưng khả năng giải thích trực tiếp từng biến còn hạn chế.